In [ ]:
from deepeval.metrics import (AnswerRelevancyMetric, 
                              ContextualPrecisionMetric, 
                              ContextualRelevancyMetric,
                              FaithfulnessMetric,
                              GEval)
from deepeval.test_case import LLMTestCaseParams, LLMTestCase
from deepeval.dataset import Golden, EvaluationDataset
from deepeval.models import DeepEvalBaseLLM
from deepevals_script import Deepeval
from dotenv import load_dotenv
load_dotenv()
import deepeval
import os
from langchain_ollama.llms import OllamaLLM
from Retrieval import Retrieval
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv
import torch
from transformers.utils import is_flash_attn_2_available
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_huggingface import HuggingFaceEmbeddings
from Pincone import Pincone_vectorStore
import warnings
from transformers import BitsAndBytesConfig
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict
from langchain_ollama import ChatOllama, OllamaLLM
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16)
use_quantization = True

In [ ]:
Device = "cuda" if torch.cuda.is_available else "cpu"
index_name="klasshour"
index= Pincone_vectorStore(index_name=index_name)
Embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2", model_kwargs= {"device":Device})
vector_store = PineconeVectorStore(embedding=Embeddings,index=index)
retrieval_obj = Retrieval(device=Device, index=index,Embeddings=Embeddings,vector_store=vector_store)
retrieval, prompt = retrieval_obj.get_retrieval()

In [ ]:
attn_implementation = "flash_attention_2" if (is_flash_attn_2_available()) and (torch.cuda.get_device_capability(0)[0]>=8) else "spda"

In [ ]:
print(f"[INFO]: using attention mechanism {attn_implementation}")
model_id = "google/gemma-2b-it"
model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id,
                                          dtype = torch.float16,
                                          quantization_config=quantization_config if use_quantization else None,
                                          device_map = "auto",
                                          attn_implementation= attn_implementation)


In [ ]:
# models = ChatOllama(
#     model="llama2:7b",
#     temperature=0.7
# )

In [ ]:
models = Deepeval(model=model, name="gemma", tokenizer=tokenizer)

In [ ]:
# model = OllamaLLM(model="llama2:7b",
#                   temperature=0.7)
# chain = (
#     {
#         "context": itemgetter("question") | retrieval,
#         "question": itemgetter("question"),
#     }
#     | prompt
#     | model
#     | StrOutputParser()
# )

# result=chain.invoke({"question": "What is Temperature"})
# print(result)

In [ ]:
API_Key= os.getenv(key="confident_AI")
deepeval.login(api_key=API_Key)

### Create a golden set evaluations dataset
1. input
2. expected_output

In [ ]:
test_data = [
    {
        "input": "If a water wave travels through water, what changes might occur that will simulate a different medium?",
        "expected_output": (
            "As a water wave travels through water, several effects may simulate a different medium: "
            "a change in amplitude due to energy loss, change in period from varying density, reflection or refraction "
            "at boundaries, and scattering or absorption depending on the medium’s properties."
        ),
    },
    {
        "input": "What are the implications of discovering many planets around nearby stars for the probability of extraterrestrial life?",
        "expected_output": (
            "The discovery of numerous exoplanets increases the likelihood of finding life beyond Earth. "
            "It supports the idea that life-friendly environments are common and motivates deeper research into "
            "detecting biosignatures and intelligent civilizations."
        ),
    },
    {
        "input": (
            "A particle's position is described by (x, y, z, t) in one inertial frame S, while another frame S′ "
            "moves with velocity v along the x-axis. Given x = x′ + vt and t = t′, how far does the particle travel "
            "in its own reference frame?"
        ),
        "expected_output": (
            "In the particle’s own reference frame, it remains at rest. Therefore, the distance traveled is zero."
        ),
    },
]


In [ ]:
goldens = []
for data in test_data:
    golden = Golden(
    input = data['input'],
    expected_output = data['expected_output']
    )
    
    goldens.append(golden)
dataset = EvaluationDataset(goldens=goldens)

In [ ]:
# dataset = EvaluationDataset()

In [ ]:
# dataset.pull("Rags_Eval_dataset")

In [ ]:
dataset

In [ ]:
for golden in dataset.goldens:
    test_cases = LLMTestCase(
        input=golden.input,
        actual_output=golden.expected_output
    )
    dataset.add_test_case(test_cases)

In [ ]:
dataset

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
QA_Chain = RetrievalQA.from_chain_type(llm=models,
                                       retriever=retrieval)

In [ ]:
response = QA_Chain("What is Temperature")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
print(response)

In [ ]:
def query_with_context(question):
    retrived = retrieval.get_relevant_documents(question)
    response = QA_Chain.run(question)
    return retrived, response

In [ ]:
actual, query = query_with_context("Principle of relativity")
actual, query

In [ ]:
from deepeval.dataset import Golden
from deepeval.test_case import LLMTestCase
from typing import List
def convert_goldens_to_test_cases(goldens: List[Golden]) -> List[LLMTestCase]:
    test_cases = []
    for golden in dataset.goldens:
        context, rag_response = query_with_context(golden.input)
        retrieval_texts = [str(content) for content in context]
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=rag_response,
            expected_output=golden.expected_output,
            retrieval_context= retrieval_texts
        )
        test_cases.append(test_case)
    return test_cases

In [ ]:
data = convert_goldens_to_test_cases(dataset)

In [ ]:
data

In [ ]:
# !deepeval set-local-model --model-name=llama2:7b --base-url="http://localhost:11434/" --api-key="ollama"

In [ ]:
deepeval.evaluate(
    data, 
    metrics= [
        deepeval.metrics.AnswerRelevancyMetric(model=model, include_reason=True),
        deepeval.metrics.FaithfulnessMetric(model=model, include_reason=True),
        deepeval.metrics.ContextualPrecisionMetric(model=model, include_reason=True),
       deepeval.metrics.ContextualRelevancyMetric(model=model, include_reason=True)
    ]
)